In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [2]:
# 已解压
# import zipfile
# import os

# # 定义解压函数
# def unzip_file(zip_src, dst_dir):
#     if zipfile.is_zipfile(zip_src):
#         fz = zipfile.ZipFile(zip_src, 'r')
#         for file in fz.namelist():
#             fz.extract(file, dst_dir)
#         print(f"解压完成，文件已保存到 {dst_dir}")
#     else:
#         print('这不是一个有效的 zip 文件！')

# # 调用解压函数
# zip_file_path = '/root/autodl-fs/daofu_detection.zip'  # 替换为你的压缩文件路径
# destination_dir = '/root/autodl-fs'     # 替换为你想要解压到的目标路径

# # 确保目标目录存在
# os.makedirs(destination_dir, exist_ok=True)

# unzip_file(zip_file_path, destination_dir)

In [3]:
# 参数配置
class Config:
    input_size = [3, 224, 224]  # 输入图片的shape
    class_dim = 2  # 分类数（倒伏/未倒伏）
    data_path = "./daofu_detection/"  # 数据集路径
    train_list_path = "./daofu_detection/train_labels2.txt"  # 训练集列表
    eval_list_path = "./daofu_detection/val_labels2.txt"  # 验证集列表
    num_epochs = 50  # 训练轮数
    batch_size = 16  # 批次大小
    learning_rate = 0.0001  # 学习率
    checkpoint_dir = "./checkpoints/"  # 模型保存路径
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# 检查可用的 GPU 设备
if torch.cuda.is_available():
    print("Available GPU devices:", torch.cuda.device_count())
    print("Current GPU device:", torch.cuda.current_device())
    print("Current GPU device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("No GPU available. Using CPU for training.")

!ls

Available GPU devices: 1
Current GPU device: 0
Current GPU device name: NVIDIA GeForce RTX 4090
checkpoints_shouge   main_daofu.ipynb	     shouge_detection.zip
daofu_detection      main_shouge.ipynb	     shouge_test_pictures
daofu_detection.zip  prediction_results.txt
daofu_test_pictures  shouge_detection


In [5]:
# 自定义数据集
class WheatDataset(Dataset):
    def __init__(self, data_path, mode='train'):
        super().__init__()
        self.data_path = data_path
        self.img_paths = []
        self.labels = []
        
        list_path = Config.train_list_path if mode == 'train' else Config.eval_list_path
        
        with open(list_path, 'r') as f:
            for line in f.readlines():
                img_path, label = line.strip().split('\t')
                self.img_paths.append(img_path)
                self.labels.append(int(label))
        
        # 数据增强和归一化
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
    
    def __getitem__(self, index):
        img_path = self.img_paths[index]
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return img, label

    def __len__(self):
        return len(self.img_paths)


In [6]:
# 定义ResNet50模型
class WheatResNet50(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.resnet = models.resnet50(pretrained=True)
        # 修改最后一层全连接层
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.resnet(x)

In [7]:
# 修改后的训练函数（只改动模型保存部分）
def train_model():
    # ...（前面的代码保持不变）
    # 创建数据集和数据加载器
    train_dataset = WheatDataset(Config.data_path, mode='train')
    eval_dataset = WheatDataset(Config.data_path, mode='eval')
    
    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True)
    eval_loader = DataLoader(eval_dataset, batch_size=Config.batch_size, shuffle=False)
    
    # 初始化模型
    model = WheatResNet50(num_classes=Config.class_dim).to(Config.device)
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=Config.learning_rate)
    
    # 训练循环
    best_acc = 0.0
    for epoch in range(Config.num_epochs):
        # ...（训练过程保持不变）
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(Config.device)
            labels = labels.to(Config.device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # 统计信息
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i+1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{Config.num_epochs}], Step [{i+1}/{len(train_loader)}], '
                      f'Loss: {loss.item():.4f}')
        
        # 每个epoch结束后在验证集上评估
        train_acc = 100 * correct / total
        eval_loss, eval_acc = evaluate_model(model, eval_loader)
        
        print(f'Epoch [{epoch+1}/{Config.num_epochs}], '
              f'Train Loss: {running_loss/len(train_loader):.4f}, '
              f'Train Acc: {train_acc:.2f}%, '
              f'Eval Loss: {eval_loss:.4f}, '
              f'Eval Acc: {eval_acc:.2f}%')
        
        # 保存最佳模型
        # 修改模型保存格式为.pt
        if eval_acc > best_acc:
            best_acc = eval_acc
            torch.save(model.state_dict(), os.path.join(Config.checkpoint_dir, 'best_model.pt'))  # 改为.pt
            print(f'Best model saved with accuracy: {best_acc:.2f}%')
    
    print('Training finished!')

In [8]:
# 评估函数
def evaluate_model(model, data_loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(Config.device)
            labels = labels.to(Config.device)
            
            outputs = model(inputs)
            loss = nn.CrossEntropyLoss()(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(data_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [9]:
# 预测函数（将结果写入txt并打印）
def predict_and_save_results(model, image_folder, output_file):
    model.eval()
    results = []
    
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])
    
    class_names = ['未倒伏', '倒伏']  # 根据实际类别顺序调整
    
    with torch.no_grad():
        for img_name in os.listdir(image_folder):
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(image_folder, img_name)
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(Config.device)
            
            output = model(img_tensor)
            prob = torch.softmax(output, dim=1)
            _, pred = torch.max(output, 1)
            
            result = {
                'filename': img_name,
                'class_id': pred.item(),
                'class_name': class_names[pred.item()],
                'probability': prob[0][pred.item()].item()
            }
            results.append(result)
            
            # 打印结果
            print(f"{img_name}: {class_names[pred.item()]} (置信度: {prob[0][pred.item()]:.2%})")
    
    # 写入txt文件
    with open(output_file, 'w') as f:
        for res in results:
            f.write(f"{res['filename']}\t{res['class_id']}\t{res['class_name']}\t{res['probability']:.4f}\n")
    
    print(f"预测结果已保存到: {output_file}")


In [10]:
if __name__ == "__main__":
    # 创建检查点目录
    os.makedirs(Config.checkpoint_dir, exist_ok=True)
    
    # 训练模型
    train_model()
    
    # 加载最佳模型进行预测（修改为加载.pt文件）
    model = WheatResNet50(num_classes=Config.class_dim).to(Config.device)
    model.load_state_dict(torch.load(os.path.join(Config.checkpoint_dir, 'best_model.pt')))  # 改为.pt
    
    # 预测测试集并保存结果
    test_images_path = "/root/autodl-fs/daofu_test_pictures/test_pictures_1/"
    output_txt = "./prediction_daofu_results.txt"
    predict_and_save_results(model, test_images_path, output_txt)

/root/miniconda3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/miniconda3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/50], Step [10/25], Loss: 0.0608
Epoch [1/50], Step [20/25], Loss: 0.0157
Epoch [1/50], Train Loss: 0.1672, Train Acc: 94.25%, Eval Loss: 0.0336, Eval Acc: 98.00%
Best model saved with accuracy: 98.00%
Epoch [2/50], Step [10/25], Loss: 0.0115
Epoch [2/50], Step [20/25], Loss: 0.0073
Epoch [2/50], Train Loss: 0.0616, Train Acc: 97.75%, Eval Loss: 0.0710, Eval Acc: 97.00%
Epoch [3/50], Step [10/25], Loss: 0.0202
Epoch [3/50], Step [20/25], Loss: 0.0818
Epoch [3/50], Train Loss: 0.0200, Train Acc: 99.50%, Eval Loss: 0.0273, Eval Acc: 99.00%
Best model saved with accuracy: 99.00%
Epoch [4/50], Step [10/25], Loss: 0.0057
Epoch [4/50], Step [20/25], Loss: 0.0536
Epoch [4/50], Train Loss: 0.0277, Train Acc: 99.25%, Eval Loss: 0.0322, Eval Acc: 99.00%
Epoch [5/50], Step [10/25], Loss: 0.0359
Epoch [5/50], Step [20/25], Loss: 0.0115
Epoch [5/50], Train Loss: 0.0135, Train Acc: 100.00%, Eval Loss: 0.0311, Eval Acc: 98.00%
Epoch [6/50], Step [10/25], Loss: 0.0019
Epoch [6/50], Step [20/25